### IPD440 - APRENDIZAJE DE MAQUINAS
# Clase 16: Large Language Models
Prof. Nicolás Torres<br/>
<a href="mailto:nicolas.torresr@usm.cl">nicolas.torresr@usm.cl</a><br/>
Ingeniería Civil Telemática<br/>
Departamento de Electrónica<br/>
Universidad Técnica Federico Santa María

# Teoría de Shannon

Shannon introdujo conceptos clave como la **información** y la **entropía** en 1948, lo que permite cuantificar y medir la cantidad de información en un mensaje y la incertidumbre asociada a este.

<div align="center">
    <img src="img/ClaudeShannon.jpg" alt="" width="15%" align="center" />
</div>

Además, mostró cómo predecir lo que viene a continuación en una secuencia, lo cual es exactamente lo que hacen los LLMs en la actualidad, basan sus respuestas en la probabilidad de las palabras siguientes en función de los datos previos.

# El Juego de Shannon 

Fue un experimento donde las personas participantes debían adivinar la siguiente palabra de una oración repetidamente. Shannon pudo utilizar el número de conjeturas para estimar los límites de la entropía de la predicción de texto a nivel humano.

In [1]:
from shannon import *

In [2]:
shannon("Life is what happens when you are busy making other plans.")

BERT: Life magazine good matters here necessary die happy working decisions decisions (0%)
GPT2: Life : like allows when you take a doing a people (20%)


In [3]:
shannon("To be or not to be that is the question.")

BERT: To me free not be be loved way impossible question (33%)
GPT2: To the clear not be be be kind to reason (22%)


# Experimento next-token prediction
¿Son mejores o peores los modelos de lenguaje en comparación a los humanos para predecir la siguiente palabra?

<div align="center">
    <img src="img/shannon_game.png" alt="" width="50%" align="center" />
</div>

# Transformers

Comencemos mirando el modelo como una sola caja negra. En una aplicación de traducción automática, tomaría una oración en un idioma y generaría su traducción en otro.

<div align="center">
    <img src="img/black-box.png" alt="" width="70%" align="center" />
</div>

Si miramos dentro de la caja negra vemos un componente de codificación, un componente de decodificación y conexiones entre ellos.

<div align="center">
    <img src="img/encoders_decoders.png" alt="" width="50%" align="center" />
</div>

El componente de codificación es una pila de codificadores y el componente de decodificación es una pila de decodificadores del mismo número.

<div align="center">
    <img src="img/stack.png" alt="" width="45%" align="center" />
</div>

Los codificadores son todos idénticos en estructura (pero no comparten pesos). Cada uno se divide en dos subcapas:

<div align="center">
    <img src="img/encoder.png" alt="" width="50%" align="center" />
</div>

La entrada del codificador pasa primero a través de una capa de autoatención, que ayuda al codificador a ver otras palabras en la oración de entrada mientras codifica una palabra específica. Veremos más de cerca la autoatención más adelante.

El decodificador tiene ambas capas, pero entre ellas hay una capa de atención que ayuda al decodificador a enfocarse en las partes relevantes de la oración de entrada.

<div align="center">
    <img src="img/blocks.jpg" alt="" width="70%" align="center" />
</div>

# Codificación

Cada palabra es un vector que alimenta al primer encoder y su salida corresponde a la entrada del siguiente.

<div align="center">
    <img src="img/encoding.png" alt="" width="40%" align="center" />
</div>

Aquí vemos una propiedad clave del Transformer: la palabra en cada posición fluye a través de su propio camino en el codificador. Existen dependencias entre estas rutas en la capa de autoatención. Sin embargo, la capa de Feed Forward no tiene esas dependencias y, por lo tanto, las diversas rutas se pueden ejecutar en paralelo mientras fluye a través de esta capa.

# Keys, Values, and Queries

El primer paso para calcular la Auto-Atención es crear tres vectores a partir de cada uno de los vectores que recibe como entrada el encoder. Entonces para cada palabra se genera un vector de consulta (q), de clave (k), y de valor (v).

<div align="center">
    <img src="img/self_attention_vectors.png" alt="" width="50%" align="center" />
</div>

# ¿Qué son estos vectores de "consulta", "clave" y "valor"?

- **Vector de Consulta (Query Vector):** $\mathbf{q} = \mathbf{X} * \mathbf{W}^Q$. Piense en esto como la palabra actual.

- **Vector de Clave (Key Vector)**: $\mathbf{k} = \mathbf{X} * \mathbf{W}^K$. Piense en esto como un mecanismo de indexación para el vector de valor. Similar a cómo tenemos pares llave-valor en un diccionario o una tabla hash, donde las llaves se usan para indexar los valores de forma única.

- **Vector de Valor (Value Vector)**: $\mathbf{v} = \mathbf{X} * \mathbf{W}^V$. Piense en esto como la información en la palabra de entrada.

# Cálculo del score

La puntuación determina cuánto enfoque colocar en otras partes de la oración de entrada a medida que codificamos una palabra en una posición determinada.

La puntuación se calcula con el producto escalar del vector de consulta con el vector clave de la palabra que estamos puntuando. Entonces, si estamos procesando la palabra en la posición #1, el primer puntaje sería el producto escalar de q1 y k1. La segunda puntuación sería el producto escalar de q1 y k2.

<div align="center">
    <img src="img/self_attention_score.png" alt="" width="50%" align="center" />
</div>

# La salida de la capa de autoatención

<div align="center">
    <img src="img/self-attention-output.png" alt="" width="50%" align="center" />
</div>

# Cálculo de la *Self-attention* a partir de q, k y v:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V $$ 

<div align="center">
    <img src="img/self-attention-calculation.png" alt="" width="50%" align="center" />
</div>

# Una bestia con muchas cabezas

Se refinó aún más la capa de autoatención al agregar un mecanismo llamado atención de “multi-headed attention”. Esto mejora el rendimiento de la capa de atención porque expande la capacidad del modelo para enfocarse en diferentes posiciones y entrega a la capa de atención múltiples “subespacios de representación”.

<div align="center">
    <img src="img/multi-heads.png" alt="" width="50%" align="center" />
</div>

# Condensar las multi-cabezas en una sola matriz
<div align="center">
    <img src="img/weight_matrix.png" alt="" width="50%" align="center" />
</div>

# Representando el orden de la secuencia

Método para explicar el orden de las palabras en la secuencia de entrada, agregando un vector a cada embedding de entrada. Estos vectores siguen un patrón específico que aprende el modelo, lo que le ayuda a determinar la posición de cada palabra o la distancia entre diferentes palabras en la secuencia.

<div align="center">
    <img src="img/positional_encoding_vectors.png" alt="" width="60%" align="center" />
</div>

# Codificadores posicionales

<div align="center">
    <img src="img/positional_encoding_example.png" alt="" width="60%" align="center" />
</div>

# Capa normalización

Un último detalle de la arquitectura del transformer es que cada codificador tiene un paso de normalización.

<div align="center">
    <img src="img/residual_layer_norm.png" alt="" width="50%" align="left" />
    <img src="img/residual_layer_norm_2.png" alt="" width="40%" align="center" />
</div>

# Decodificación

El decoder comienza procesando la secuencia de entrada. La salida del codificador superior se transforma luego en un conjunto de vectores de atención K y V, que son utilizados para enfocarse en los lugares apropiados en la secuencia de entrada.

<div align="center">
    <img src="img/decoder.png" alt="" width="50%" align="center" />
</div>

# La capa de salida


El vector producido por el decoder ingresa a una capa lineal (FC) que proyecta la entrada a un vector mucho más grande llamado logits con el tamaño del vocabulario. Cada entrada de este vector corresponde al puntaje de cada palabra en el vocabulario. Finalmente, la capa softmax convierte esos puntajes en probabilidades (todos positivos y suman 1.0). Se elige la celda con la probabilidad más alta y la palabra asociada a ella se entrega como salida.

<div align="center">
    <img src="img/decoder_output.png" alt="" width="40%" align="center" />
</div>

# Arquitectura del Transformer
<div align="center">
    <img src="img/transformer.png" alt="" width="70%" align="center" />
</div>

# Generative Pretraining Transformer (GPT)

OpenAI propuso el primer modelo GPT en 2018. La principal contribución de GPT fue entrenar previamente **un** modelo de lenguaje general y usarlo directamente para **varias** tareas.

# Arquitectura GPT

Utiliza 12 decodificadores para producir los embeddings de entrada. Las secuencias de entrada se codifican mediante la codificación de pares de bytes (BPE), un algoritmo de compresión de datos adecuado para la segmentación de palabras que permite codificar palabras raras y fuera del vocabulario (OOV). Además de los BPE, también se codifica la posición de cada token de entrada.

<div align="center">
    <img src="img/arquitecturaGPT.png" alt="" width="25%" align="center" />
</div>

# Entrenamiento

El conjunto de datos de BooksCorpus que consta de 7000 libros no publicados se utilizó como datos de entrenamiento para dos tareas: 1) predicción de texto, es decir, predecir el siguiente token a partir del actual, y 2) clasificación de texto.

Después de entrenar el modelo de lenguaje, GPT se ajusta (fine-tuning) para cumplir cuatro nuevas tareas.

## Inferencia de lenguaje natural 
Esta tarea es para detectar la relación entre oraciones. Se utilizan cinco conjuntos de datos diferentes, incluidos Stanford Natural Language Inference, ficción popular e informes gubernamentales, para ajustar el modelo GPT.

## Respuesta a preguntas y razonamiento de sentido común 

Esta tarea se utiliza para inculcar la capacidad de comprender información contextual de largo alcance en el modelo. Aquí, se utiliza el conjunto de datos de comprensión de lectura de los exámenes (RACE), que consta de pasajes en inglés con preguntas de exámenes de secundaria y preparatoria. El modelo también se evalúa en el conjunto de datos Story Cloze Test, donde el final de una historia corta debe elegirse entre dos opciones.

## Similitud semántica 
La similitud semántica o la detección de paráfrasis es una tarea desafiante que requiere una comprensión de la negación, la detección de conceptos reformulados y la ambigüedad. Se utilizan tres conjuntos de datos para esta tarea: Microsoft Paraphrase corpus (MRPC), que consta de fuentes de noticias, el conjunto de datos de pares de preguntas de Quora (QQP) y el punto de referencia de similitud textual semántica (STS).

## Clasificación
Se consideran dos tareas de clasificación, una para detectar si una oración es gramaticalmente correcta o no, una excelente manera de medir los sesgos lingüísticos innatos en el modelo, y la otra para la clasificación de sentimientos. Los conjuntos de datos Corpus of Linguistic Acceptability (CoLA) y Stanford Sentiment Treebank (SST-2) se utilizan para cada tarea, respectivamente.

In [2]:
# pip install transformers

In [3]:
from transformers import pipeline, logging

logging.set_verbosity_error()

In [4]:
GPT = pipeline('text-generation', model='openai-gpt')

In [4]:
gen_text = GPT("A golden retriever is a", max_length=100, num_return_sequences=5)

In [5]:
for k in gen_text:
    print(k['generated_text'].replace('\n',''),"\n")

A golden retriever is a wolf. "  " is that right? " i whispered.  " it is, " he answered.  i was looking over ty and in my head i was trying to figure out if he had the same story of his mom that i did. we 'd worked together for three years as a team, my boss was known to be in the business but if it was a case of being an accountant, something he called a'boy genius'he didn't have a 

A golden retriever is a beautiful sight, no? " he said. he turned his head and winked at me. " i think it's time i take this puppy to town and show her something. "  " fine. "  he started to walk away and my mouth fell open as i looked into his face, " why are you taking the dog? "  he tilted his head and said, " oh, nothing for you to worry about, mr. moore. i'm taking the dog to 

A golden retriever is a little odd, but i like to make a good living, " she said.  he shrugged, a gesture that she appreciated. her mother had given her a lot of bad tips. his smile widened. his eyes brightened. he 'd found o

In [6]:
gen_text = GPT("A golden retriever is not a", max_length=100, num_return_sequences=5)

In [7]:
for k in gen_text:
    print(k['generated_text'].replace('\n',''),"\n")

A golden retriever is not a good enough reason. it is all about appearances. he is to be taken care of - as that means that he needs to be cared for. "  " so will you give him to me? "  " no. i do not plan on letting you have him. "  " but... you promised. a puppy... "  " but even if we have one, i do not foresee that he will need that kind of care. "  " but you 

A golden retriever is not a threat. "  " i can not accept you to be my mistress, " lord newlyn hissed, and he took the knife away.  " i will accept nothing but the best from this man, and i would not see anything of the likes of him again, " rose said and, not wanting to be insulted, began to wipe at the tears on her cheeks, but he was not at all satisfied. " i must leave. "  her words carried on 

A golden retriever is not a pet. it's a dog. "  i felt a twinge of disappointment. i didn't expect anyone else to be able to hear my thoughts, but the dog spoke, " i'm not a pet. and i hope you know i 'd be happy to lick whatever h

# Limitación GPT-1

GPT fue un gran éxito y superó a los modelos anteriores en la mayoría de las tareas. Sin embargo, una limitación de este modelo es su naturaleza unidireccional porque el modelo solo se entrenó para predecir la siguiente palabra a partir de la palabra actual y no al revés.

# Bidirectional Encoder Representations from Transformers (BERT)

Google AI introdujo un modelo de lenguaje basado en codificador que, a diferencia de GPT, se entrena en ambas direcciones. En el artículo se investigan dos versiones de este modelo, BERT_BASE, que tiene el tamaño de GPT, y un modelo más grande, BERT_LARGE, con 340 M de parámetros y 24 bloques de Transformers. BooksCorpus y Wikipedia en inglés se usan para entrenar previamente al modelo en dos tareas: modelo de lenguaje enmascarado y predicción de la siguiente oración.

Las entradas del modelo BERT están codificadas de una manera específica que consta de tres partes: (1) **WordPiece tokenization embeddings** que descompone las palabras en partes más pequeñas y las convierte en vectores, (2) **Segment embeddings** para identificar qué tokens pertenecen a la primera oración y cuáles pertenecen a la segunda oración, y (3) **Position embeddings** permite una ventana de contexto 512, es decir, posiciones 0–511. 

<div align="center">
    <img src="img/BERT.png" alt="" width="60%" align="center" />
</div>

# Variaciones de BERT 

Existen muchas variaciones del modelo BERT. Un Lite BERT (ALBERT), por ejemplo, es una versión liviana de BERT donde la cantidad de parámetros (x18 menos parámetros) y el tiempo de entrenamiento (x1.7 más rápido) se reducen. 

El enfoque BERT robustamente optimizado (RoBERTa) es otra variación en la que las mejoras se realizan esencialmente entrenando BERT en un conjunto de datos más grande con lotes más grandes. 

Existen otras variaciones para diferentes idiomas, como BETO en español.

# Limitación BERT

BERT muestra un gran rendimiento con su naturaleza bidireccional; sin embargo, parece quedarse corto en la comprensión de la negación.

In [5]:
BERT = pipeline('fill-mask', model='bert-base-uncased')

In [9]:
BERT('A golden retriever is a [MASK].')

[{'score': 0.7484621405601501,
  'token': 3899,
  'token_str': 'dog',
  'sequence': 'a golden retriever is a dog.'},
 {'score': 0.044763416051864624,
  'token': 13314,
  'token_str': 'mascot',
  'sequence': 'a golden retriever is a mascot.'},
 {'score': 0.023076163604855537,
  'token': 4743,
  'token_str': 'bird',
  'sequence': 'a golden retriever is a bird.'},
 {'score': 0.020245300605893135,
  'token': 4937,
  'token_str': 'cat',
  'sequence': 'a golden retriever is a cat.'},
 {'score': 0.016355212777853012,
  'token': 8843,
  'token_str': 'breed',
  'sequence': 'a golden retriever is a breed.'}]

In [10]:
BERT('A golden retriever is not a [MASK].')

[{'score': 0.7157089114189148,
  'token': 3899,
  'token_str': 'dog',
  'sequence': 'a golden retriever is not a dog.'},
 {'score': 0.03916870057582855,
  'token': 13314,
  'token_str': 'mascot',
  'sequence': 'a golden retriever is not a mascot.'},
 {'score': 0.016858205199241638,
  'token': 8843,
  'token_str': 'breed',
  'sequence': 'a golden retriever is not a breed.'},
 {'score': 0.01580881141126156,
  'token': 4937,
  'token_str': 'cat',
  'sequence': 'a golden retriever is not a cat.'},
 {'score': 0.013033213093876839,
  'token': 4743,
  'token_str': 'bird',
  'sequence': 'a golden retriever is not a bird.'}]

# GPT-2

GPT-2, el sucesor de GPT fue presentado en 2019 por OpenAI. En su propuesta original, se entrenan diferentes modelos con diferentes tamaños. El más pequeño tiene el mismo tamaño que GPT, el segundo más pequeño tiene el mismo tamaño que el modelo BERT más grande y el más grande con 1542M de parámetros es aproximadamente 10 veces más grande que GPT (con 48 bloques decodificadores y más dimensiones de salida). Los datos de entrenamiento se descargan desde 8 millones de páginas web en Reddit para completar un total de 40 GB de texto.

# GPT-2 vs BERT

GPT-2 supera a BERT en siete de las ocho tareas de NLP de referencia. Este gran modelo muestra que el rendimiento aumenta con más parámetros entrenables. Sin embargo, tener una arquitectura más compleja no es la única diferencia entre GPT-2 y los modelos anteriores. La principal diferencia es que el condicionamiento de tareas se usa durante el entrenamiento del modelo de lenguaje para eliminar la necesidad de ajustes (fine-tuning).

# Condicionamiento de tareas

El condicionamiento de tareas es un objetivo de aprendizaje modificado en el que se entrena el modelo de lenguaje para encontrar la probabilidad de la salida dada la entrada y la tarea (o indicación), es decir, $P(salida | entrada, tarea)$ en lugar de solo $P(salida | entrada)$. Esto permite que el modelo produzca una salida diferente dada la misma entrada para diferentes tareas. El acondicionamiento de tareas es lo que permite que GPT-2 pueda producir resultados precisos en tareas en las que no ha sido entrenado previamente al proporcionar solo un aviso. Por ejemplo, para realizar resúmenes, se agrega “TL;DR:” al texto original.

GPT-2 ofrece resultados impresionantes con tareas como la comprensión de lectura, pero proporciona resultados rudimentarios para algunas tareas como el resumen de texto.

In [6]:
GPT2 = pipeline('text-generation', model='gpt2')

In [12]:
gen_text = GPT2('A golden retriever is a', num_return_sequences=5, 
                     max_length=100, pad_token_id=GPT2.tokenizer.eos_token_id)

In [13]:
for k in gen_text:
    print(k['generated_text'].replace('\n',''),"\n")

A golden retriever is a good companion, but if you're looking for a more advanced companion, get another in your pet's litter box. With your favorite dogs you'll never be virtuoso. Get yourself one in your litter box to give it a try, and get started.This will become your last companion; at any time you can give it to other pets, as they like to do. Once you have your friend, take him away.Beware if your 

A golden retriever is a miniature cat with a nose, and an all clear head. She is about 6 feet tall with a skinny neck. She is known for her sharp talons, and long fingernails that she wears, which seem to be part of her costume—usually from her fur. Her wingspan is about 15 feet, so if you've ever seen a golden retriever, you will understand. It has short, white wings.Advertisement Continue reading the main story 

A golden retriever is a little rare -- it breeds on farms all over the world so it usually hasn't been bred in England or France to grow food crops. But there's a reason w

In [14]:
gen_text = GPT2('A golden retriever is not a', num_return_sequences=5, 
                     max_length=100, pad_token_id=GPT2.tokenizer.eos_token_id)

In [15]:
for k in gen_text:
    print(k['generated_text'].replace('\n',''),"\n")

A golden retriever is not a special breed. The golden retriever is generally a purebred gray to green puppy. If yours is a gray to green puppy, however, this is likely where you are, because the genetics of your dog are affected, and that is just your genetics. As you breed to be a golden retriever, you will need to find out what you want and how to breed. Your best bet is to work out how many breeding trips you'll need. As with many 

A golden retriever is not a great dog. It may be too small and have some markings like a "pinkie," a bit of an "unshakably strong gray," a spot where he can get lost. A Golden Retret is not a good orator, and has bad posture so his posture will be disturbed or he'll get hurt. If used at all, he will be treated like anything else. A golden retriever is a good trainer. There may be times where 

A golden retriever is not a pet. It is the pet. It is the companion.I feel like I'm running out of water. I still have to pee, and I'm not happy in the least with 

# GPT-3

OpenAI introdujo GPT-3 en 2020, que es 100 veces más grande que GPT-2 con 175 billones de parámetros. Este modelo comparte la misma arquitectura con GPT-2 con más capas de decodificación (96 bloques frente a 48 bloques) y más dimensiones. Para el entrenamiento se utiliza una combinación ponderada de cinco conjuntos de datos, Common Crawl, WebText2, Books1, Books2 y Wikipedia. El modelo se evalúa en un entorno de aprendizaje de pocas tomas en varias tareas. El aprendizaje de pocos intentos es cuando se proporcionan algunos ejemplos además de la descripción de la tarea como indicación. Esto es similar a cómo los humanos aprenden nuevas tareas de lenguaje con solo unos pocos ejemplos. El modelo siguió mejorándose con diferentes iteraciones conocidas como la serie GPT-3.5, incluido ChatGPT, centrado en la conversación.

In [16]:
# pip install openai

In [4]:
from ChatGPT import GPT

In [13]:
GPT('A golden retriever is a', model="gpt-3.5-turbo", num_return_sequences=5, max_length=100)

A golden retriever is a popular breed of dog known for their friendly and gentle temperament. They are often used as service dogs, therapy dogs, and search and rescue dogs due to their intelligence and loyalty. Golden retrievers have a distinctive golden coat that is water-repellent and they are known for their love of water and retrieving objects. They are great family pets and are known for being good with children. 

A golden retriever is a popular and friendly dog breed known for its intelligence, loyalty, and gentle temperament. They are often used as guide dogs, therapy dogs, and search and rescue dogs due to their excellent trainability and friendly nature. Golden retrievers have a distinctive golden coat that is dense and water-repellent, making them excellent swimmers. They are great family pets and are known for their love of people, especially children. 

A golden retriever is a popular breed of dog known for its friendly and gentle temperament. They are intelligent, loyal, 

In [14]:
GPT('A golden retriever is not a ...', model="gpt-3.5-turbo", num_return_sequences=5, max_length=100)

A golden retriever is not a fish. 

A golden retriever is not a cat. 

A golden retriever is not a type of cat. 

A golden retriever is not a type of fish. 

A golden retriever is not a cat. 



# Limitaciones GPT-3
GPT-3 es capaz de escribir artículos de noticias persuasivos y producir código en LaTeX, SQL, etc., y se muestra que su rendimiento es comparable al de los modelos BERT perfeccionados. Sin embargo, carece de coherencia a medida que el texto comienza a alargarse. Además, las oraciones se repiten a medida que aumenta la longitud del texto generado. Esto puede deberse a su naturaleza unidireccional.

# GPT-4

OpenAI introdujo GPT-4 en 2023, un avance significativo respecto a GPT-3. Aunque el número exacto de parámetros de GPT-4 no ha sido revelado, se sabe que es considerablemente más grande que los 175 mil millones de parámetros de GPT-3. Al igual que su predecesor, GPT-4 mantiene una arquitectura basada en transformadores, pero con optimizaciones en la gestión de la memoria y mayor capacidad para manejar contextos más largos, permitiendo procesar hasta 25,000 tokens en comparación con los 4,096 tokens de GPT-3. GPT-4 ha sido entrenado con un conjunto de datos más diverso y refinado, abarcando una variedad más amplia de dominios y fuentes de información. Este modelo se destaca por mejorar el rendimiento en tareas de aprendizaje de pocos disparos, en las que es capaz de interpretar instrucciones complejas con pocos ejemplos. Además, introduce capacidades multimodales, permitiendo procesar no solo texto, sino también imágenes, lo que lo diferencia aún más de GPT-3 en términos de versatilidad y comprensión contextual.

In [15]:
GPT('A golden retriever is a', model="gpt-4o-mini", num_return_sequences=5, max_length=100)

A Golden Retriever is a breed of dog known for its friendly, intelligent, and loyal nature. They are medium to large-sized dogs with a dense, water-repellent outer coat and a thick undercoat, which can be various shades of golden. Golden Retrievers are often used as service dogs, therapy dogs, and in search and rescue due to their intelligence and eagerness to please.

They are known for their playful and gentle temperament, making them excellent family pets. They require regular exercise and 

A Golden Retriever is a breed of dog known for its friendly, intelligent, and devoted nature. They are medium to large-sized dogs with a dense, water-repellent double coat that is typically golden in color. Golden Retrievers are highly regarded for their gentle temperament, making them excellent family pets and companions.

Originally bred in Scotland in the late 19th century for retrieving game during hunting, they are now popular not only as pets but also as service dogs, therapy dogs, and in 

In [16]:
GPT('A golden retriever is not a ...', model="gpt-4o-mini", num_return_sequences=5, max_length=100)

A golden retriever is not a guard dog. While they are friendly, intelligent, and loyal, they are not typically bred for protective instincts like some other breeds. Instead, they are known for their gentle temperament and are often used as therapy and service dogs. 

A golden retriever is not a guard dog. While they are friendly, loyal, and intelligent, their temperament is generally more suited for companionship and service roles rather than protection. 

A golden retriever is not a guard dog. While they are friendly, loyal, and protective of their families, they are not typically aggressive or territorial like some breeds that are specifically bred for guarding. Golden retrievers are known for their friendly demeanor and are often used as therapy and service dogs. 

A golden retriever is not a guard dog. While they are friendly, loyal, and intelligent, they are not typically bred or trained for protection or guarding purposes. Instead, they are known for their gentle temperament and 